<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed):
    # Carregar o dataset
    X, y = fetch_openml(
        name="mnist_784",
        version=1,
        as_frame=False,
        return_X_y=True
    )
    
    # Converter rótulos para inteiro
    y = y.astype(int)
    
    # Normalizar (boa prática)
    X = X / 255.0
    
    # Divisão treino/teste com reprodutibilidade
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=10000,
        random_state=seed,
        stratify=y
    )
    
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = load_data(seed=42)

print(X_train.shape)
print(X_test.shape)

(60000, 784)
(10000, 784)


# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

def train_random_forest(X_train, y_train, seed):
    model = RandomForestClassifier(
        n_estimators=100,      # número de árvores
        random_state=seed,
        n_jobs=-1              # usa todos os núcleos (mais rápido)
    )
    
    model.fit(X_train, y_train)
    
    return model

def train_adaboost(X_train, y_train, seed):
    model = AdaBoostClassifier(
        n_estimators=100,      # número de estimadores
        random_state=seed
    )
    
    model.fit(X_train, y_train)
    
    return model



rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [5]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    # Fazer previsões
    y_pred = model.predict(X_test)
    
    # Calcular acurácia
    acc = accuracy_score(y_test, y_pred)
    
    return acc

rf_acc = evaluate(rf_model, X_test, y_test)
ab_acc = evaluate(ab_model, X_test, y_test)

print("Random Forest:", rf_acc)
print("AdaBoost:", ab_acc)

Random Forest: 0.9685
AdaBoost: 0.7313


**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [6]:
def run_pipeline(model_type="rf", seed=42):
    # 1. Carregar dados
    X_train, X_test, y_train, y_test = load_data(seed)
    
    # 2. Escolher modelo
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
    # 3. Avaliar modelo
    acc = evaluate(model, X_test, y_test)
    
    return acc

rf_acc = run_pipeline(model_type="rf", seed=42)
ab_acc = run_pipeline(model_type="ab", seed=42)

print("Random Forest:", rf_acc)
print("AdaBoost:", ab_acc)

Random Forest: 0.9685
AdaBoost: 0.7313


**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_full(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="macro"),
        "recall": recall_score(y_test, y_pred, average="macro"),
        "f1": f1_score(y_test, y_pred, average="macro")
    }

# Carregar dados
X_train, X_test, y_train, y_test = load_data(seed=42)

# Treinar modelos
rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

# Avaliar
rf_results = evaluate_full(rf_model, X_test, y_test)
ab_results = evaluate_full(ab_model, X_test, y_test)

print("Random Forest:", rf_results)
print("AdaBoost:", ab_results)

Random Forest: {'accuracy': 0.9685, 'precision': 0.9682659058959411, 'recall': 0.9682233505147669, 'f1': 0.9682285118146889}
AdaBoost: {'accuracy': 0.7313, 'precision': 0.7436588173477231, 'recall': 0.7278016101722219, 'f1': 0.7304053629628011}


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
# Seed 42
rf_42 = run_pipeline("rf", seed=42)
ab_42 = run_pipeline("ab", seed=42)

# Seed 7
rf_7 = run_pipeline("rf", seed=7)
ab_7 = run_pipeline("ab", seed=7)

print("RF (42):", rf_42)
print("RF (7):", rf_7)

print("AB (42):", ab_42)
print("AB (7):", ab_7)

Ao executar o pipeline com diferentes seeds, observou-se uma pequena variação nos resultados. Isso ocorre porque a divisão dos dados em treino e teste depende de aleatoriedade, assim como o treinamento de alguns modelos.

No entanto, o experimento é reprodutível, pois ao utilizar a mesma seed, os resultados obtidos são consistentes e podem ser replicados. Portanto, o controle da aleatoriedade por meio do parâmetro random_state garante a reprodutibilidade do experimento.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
from sklearn.metrics import accuracy_score

# Treino
y_train_pred = rf_model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)

# Teste
y_test_pred = rf_model.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)

print("Acurácia treino:", train_acc)
print("Acurácia teste:", test_acc)

Ao comparar a acurácia no conjunto de treino e teste, observa-se que o modelo apresenta desempenho significativamente superior no treino, indicando a presença de overfitting. Isso significa que o modelo está aprendendo padrões específicos dos dados de treino, mas não generaliza tão bem para novos dados.

Entre os modelos analisados, o Random Forest tende a sofrer mais com overfitting, pois possui alta capacidade de modelagem e pode memorizar os dados de treino. Já o AdaBoost, por utilizar modelos mais simples e focar em erros progressivamente, tende a ser menos suscetível a esse problema em um estágio inicial.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
import pandas as pd

n_values = [10, 50, 100, 200]

rf_results = []
ab_results = []

for n in n_values:
    # Random Forest
    rf = train_random_forest(X_train, y_train, seed=42)
    rf.set_params(n_estimators=n)
    rf.fit(X_train, y_train)
    
    rf_acc = evaluate(rf, X_test, y_test)
    rf_results.append((n, rf_acc))
    
    # AdaBoost
    ab = train_adaboost(X_train, y_train, seed=42)
    ab.set_params(n_estimators=n)
    ab.fit(X_train, y_train)
    
    ab_acc = evaluate(ab, X_test, y_test)
    ab_results.append((n, ab_acc))

print("Random Forest:", rf_results)
print("AdaBoost:", ab_results)

Ao variar o hiperparâmetro n_estimators, observou-se que o desempenho do Random Forest melhora inicialmente com o aumento do número de árvores, mas rapidamente se estabiliza, apresentando ganhos marginais.

Já o AdaBoost apresentou maior variação de desempenho conforme o número de estimadores foi alterado, indicando maior sensibilidade a esse hiperparâmetro.

Portanto, o modelo AdaBoost é mais sensível a mudanças em n_estimators, enquanto o Random Forest tende a ser mais estável devido à natureza independente de suas árvores.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. A acurácia é suficiente para avaliar os modelos?

Não, a acurácia sozinha não é suficiente, especialmente em problemas de classificação multiclasse como o Fashion MNIST. Ela mede apenas a proporção de acertos totais, mas não revela como o modelo está se comportando em cada classe individualmente, podendo mascarar erros importantes.

Por isso, é essencial considerar outras métricas como precisão, recall e F1-score, que fornecem uma visão mais completa do desempenho do modelo, especialmente em cenários onde pode haver confusão entre classes semelhantes.

2. Como você garante que o resultado não ocorreu por acaso?

A utilização de uma seed (random_state) garante reprodutibilidade, ou seja, os mesmos resultados podem ser obtidos ao repetir o experimento nas mesmas condições. Isso reduz o impacto da aleatoriedade na divisão dos dados e no treinamento dos modelos.

Além disso, testar diferentes seeds e observar que os resultados permanecem consistentes (com pequenas variações) aumenta a confiança de que o desempenho observado não ocorreu por acaso, mas sim reflete o comportamento real do modelo.

3. Cite dois possíveis problemas metodológicos neste experimento.

Um possível problema é a utilização de apenas uma divisão treino/teste, o que pode não representar bem a variabilidade dos dados. O ideal seria utilizar validação cruzada para obter uma estimativa mais robusta do desempenho do modelo.

Outro problema é a falta de ajuste fino de hiperparâmetros. Os modelos foram treinados com configurações padrão ou pouco exploradas, o que pode limitar seu desempenho e levar a conclusões incompletas sobre qual modelo é realmente melhor.

4. O pipeline implementado é confiável? Justifique.

Sim, o pipeline é confiável do ponto de vista de organização e reprodutibilidade, pois separa claramente as etapas de carregamento, treinamento e avaliação, além de controlar a aleatoriedade com o uso de seeds.

No entanto, sua confiabilidade pode ser limitada do ponto de vista experimental, já que não inclui validação cruzada nem uma exploração mais ampla de hiperparâmetros. Portanto, ele é adequado para análises iniciais, mas pode ser melhorado para avaliações mais rigorosas.